# Camera-trap live demonstration
Run both cells in a fresh Colab runtime with the team Drive mounted. The default batch uses all team-collected test images and optionally uploaded new photos. The CCT20 test batch contains the first 15 images by ID. Choose photos for individual events or video to sample the first five seconds as one event.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
PROJECT = Path.cwd()
if not (PROJECT/'runtime/live.py').is_file():
    PROJECT = Path(tempfile.mkdtemp())/'project'
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','project','https://github.com/salehdev3/Capstone_5.git',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
sys.path.insert(0,str(PROJECT/'runtime'))
from live import cases, execute, initialize, video_cases, team_test_cases, cct_test_cases
MODEL = 'project_release' #@param ['project_release','local_coco']
if MODEL == 'local_coco':
    from general_animals import GeneralAnimals
    pipeline = GeneralAnimals()
else:
    pipeline, decision = initialize(ROOT)
print('Loaded:', MODEL, pipeline.policy_sha256)


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage
DEMO_INPUT = 'team_test' #@param ['team_test','cct_test','validation','photos','video']
ADD_NEW_PHOTOS = True #@param {type:"boolean"}
if DEMO_INPUT in ('photos','video'):
    from google.colab import files
    from uuid import uuid4
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Select a file to demonstrate.')
    upload_dir = Path(tempfile.mkdtemp())
    batch = []
    if DEMO_INPUT == 'video':
        if len(uploaded) != 1:
            raise ValueError('Select one video.')
        name,raw = next(iter(uploaded.items()))
        clip = upload_dir/Path(name).name
        clip.write_bytes(raw)
        batch = video_cases(clip,upload_dir/'frames')
    for index,(name,raw) in enumerate(uploaded.items() if DEMO_INPUT == 'photos' else []):
        path = upload_dir/f'{index}.jpg'
        path.write_bytes(raw)
        batch.append((path,dict(id=f'{index}-{Path(name).stem}',event=f'upload-{index}')))
elif DEMO_INPUT == 'cct_test':
    batch = cct_test_cases(ROOT)
elif DEMO_INPUT == 'team_test':
    batch = team_test_cases(ROOT)
    if ADD_NEW_PHOTOS:
        from google.colab import files
        print('Upload new team photos, or cancel to use the existing test images.')
        uploaded = files.upload()
        upload_dir = Path(tempfile.mkdtemp())
        for index, (name, raw) in enumerate(uploaded.items()):
            path = upload_dir / f'new_{index}.jpg'
            path.write_bytes(raw)
            batch.append((path, dict(id=f'new-{index}-{Path(name).stem}',
                                     event=f'new-upload-{index}')))
else:
    batch = cases(ROOT)
print('Images selected:', len(batch))
dest, result = execute(pipeline,batch,ROOT/'presentation_demos')
display(pd.DataFrame(result['events']))
display(pd.DataFrame([{'id': f['id'], 'animals': f['count'], 'inference_seconds': f['timing']['total']} for f in result['frames']]))
for index, frame in enumerate(result['frames'],1):
    display(DisplayImage(filename=str(dest/f'frame_{index:02}.jpg'),width=720))
print('HTTP response and recorded replay:',dest)
